# <img style="float: left; padding-right: 10px; width: 100px" src="https://raw.githubusercontent.com/sbildea/SEED/main/imagini/unitbv_mie.png"> <center>Știința Explorării și Exploatării Datelor - SEED1</center>


##    <center>Seminar 2a: Dataframe cheat-sheet</center>
**Universitatea Transilvania**<br>
**SIIPA, Semestrul I**<br>
**Instructor:** Conf. dr. Teodor &Scedil;tefan B&icirc;ldea<br>
**Contact:** teodor.bildea@unitbv.ro<br>

### Cuprins
0. Recapitulare laborator 1
1. Citirea fisierelor 
2. Combinarea mai multor dataframe-uri
3. Lucrul cu dataframe-uri
4. Vizualizarea datelor: matplotlib, plotly

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sbildea/SEED/blob/main/cursuri/02-explorare-vizualizare/seminar-02a-dataframe.ipynb)

In [ ]:
# Configurare curs SEED: rulați această celulă prima.
# În Colab descarcă materialele cursului și instalează pachetele lipsă (aprox. un minut).
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/SEED"):
        !git clone -q --depth 1 https://github.com/sbildea/SEED.git /content/SEED
        !pip install -q -r /content/SEED/requirements.txt
    %cd /content/SEED/cursuri/02-explorare-vizualizare
sys.path.insert(0, os.path.abspath("../.."))

## La ce folosește acest notebook

Nu scrieți cod de la zero: îi cereți codul unui asistent AI. Ca să cereți bine, trebuie să știți **ce operații există** și **cum se numesc**. Acesta este vocabularul: pentru fiecare operație găsiți (1) ce face, (2) cum o formulați într-o cerere, (3) un exemplu mic care rulează. Rulați celulele, schimbați valorile și observați ce se întâmplă.

Reguli de aur pentru cereri: spuneți **numele tabelului și al coloanelor**, **ce vreți să obțineți** și **ce verificare vreți la final** (de exemplu „afișează câte rânduri au rămas”).

## Glosar: cuvântul din cerere → operația pandas

| Spuneți în cerere | Operația | Secțiunea |
|---|---|---|
| „citește fișierul / încarcă setul” | `read_csv`, `read_excel`, `incarca` | 1 |
| „câte rânduri și coloane, ce tipuri, valori lipsă” | `shape`, `info`, `describe`, `isna` | 2 |
| „păstrează doar rândurile care…” | filtrare cu condiție (`&`, `\|`, `isin`) | 3 |
| „adaugă o coloană calculată” | `assign`, operații pe coloane | 4 |
| „ordonează după…” | `sort_values` | 5 |
| „pe fiecare… calculează suma/media/numărul” | `groupby` + agregare, `value_counts` | 6 |
| „unește cele două tabele după cheia…” | `merge`, `concat` | 7 |
| „elimină / completează valorile lipsă” | `dropna`, `fillna` | 8 |
| „elimină duplicatele” | `drop_duplicates` | 9 |
| „împarte în grupe / intervale” | `pd.cut`, `pd.qcut`, discretizare cu coloană de referință | 10 |

In [ ]:
import pandas as pd
import numpy as np

# tabel mic pentru exemple
df = pd.DataFrame({
    'Nume': ['Ana', 'Ion', 'Maria', 'Elena', 'Mihai', 'Dan'],
    'Varsta': [25, 30, 22, 28, 35, 31],
    'Departament': ['IT', 'HR', 'IT', 'Vanzari', 'IT', 'HR'],
    'Salariu': [50000, 60000, 45000, 55000, 70000, 60000],
})
df

## 1. Citirea datelor

**Cerere:** *„Încarcă setul `lab-set4` și arată primele 5 rânduri.”*

- seturile mari ale cursului se încarcă **numai** cu `incarca` (fără căi, fără descărcări manuale);
- `incarca` întoarce un dicționar nume_tabel → DataFrame; primul tabel este `[0]`;
- fișierele proprii: `pd.read_csv("fisier.csv")`, `pd.read_excel("fisier.xlsx", sheet_name="foaie")`.

In [ ]:
from seed.date import incarca, lista_seturi

date = incarca("lab-set1")      # dicționar: nume tabel -> DataFrame
print(list(date.keys()))
clienti = date[0]               # primul tabel (sau după nume)
clienti.head()

## 2. Primul contact cu un tabel

**Cerere:** *„Câte rânduri și coloane are tabelul, ce tip are fiecare coloană, câte valori lipsă, câte valori distincte, statistica coloanelor numerice?”*

In [ ]:
print(clienti.shape)             # (rânduri, coloane)
clienti.info()                   # tipuri și valori nenule pe coloană
clienti.describe()               # statistică pentru coloanele numerice

In [ ]:
clienti.isna().sum()                       # valori lipsă pe coloană
clienti['Contract'].nunique()              # câte valori distincte
clienti['Contract'].value_counts()         # frecvența fiecărei valori

## 3. Selectare și filtrare

**Cerere:** *„Păstrează doar angajații din IT cu vârsta peste 25 de ani; arată câte rânduri rămân.”*

- coloane: `df['Nume']` (una) sau `df[['Nume', 'Salariu']]` (mai multe);
- condiții: `>`, `<`, `==`, `!=`; combinate cu `&` (și), `|` (sau), fiecare condiție între paranteze;
- `isin([...])` pentru „una dintre valorile…”.

In [ ]:
df[df['Varsta'] > 25]
df[(df['Departament'] == 'IT') & (df['Varsta'] > 25)]
df[df['Departament'].isin(['IT', 'HR'])]

## 4. Coloane noi

**Cerere:** *„Adaugă coloana `Venit_anual` = salariu × 12 și coloana `Procent` = ponderea salariului fiecărui angajat în totalul salariilor.”*

In [ ]:
df['Venit_anual'] = df['Salariu'] * 12
df['Procent'] = df['Salariu'] / df['Salariu'].sum() * 100
df

**Verificare:** procentele trebuie să însumeze 100: `df['Procent'].sum()`.
Pentru date calendaristice: `pd.to_datetime(df['Data'])`, apoi `df['Data'].dt.month`, `df['Data'].dt.year`, `df['Data'].dt.to_period('M')` (luna).

## 5. Sortare

**Cerere:** *„Ordonează descrescător după salariu și arată primii 3.”*

In [ ]:
df.sort_values(by='Salariu', ascending=False).head(3)

## 6. Grupare și agregare

**Cerere:** *„Pe fiecare departament, calculează numărul de angajați, salariul mediu și salariul total.”*

Structura gândirii: **după ce grupez** (`groupby`), **ce coloană** agreg, **cu ce funcție** (`sum`, `mean`, `count`, `min`, `max`, `median`).

In [ ]:
df.groupby('Departament').agg(
    angajati=('Nume', 'count'),
    salariu_mediu=('Salariu', 'mean'),
    salariu_total=('Salariu', 'sum'),
).reset_index()

In [ ]:
# tabel încrucișat: câți angajați pentru fiecare combinație departament × vârstă
pd.crosstab(df['Departament'], df['Varsta'])

## 7. Combinarea tabelelor

**Cerere:** *„Adaugă în tabelul angajaților managerul fiecărui departament, folosind tabelul `manageri` și cheia `Departament`.”*

- `merge` unește **după o cheie** (ca VLOOKUP): `how='left'` păstrează toate rândurile din tabelul din stânga;
- `pd.concat([t1, t2])` pune tabelele **unul sub altul** (aceleași coloane).

**Verificare:** numărul de rânduri trebuie să rămână 6; rândurile cu `Manager` lipsă arată chei care nu s-au potrivit (aici `Vanzari`).

In [ ]:
manageri = pd.DataFrame({'Departament': ['IT', 'HR'], 'Manager': ['Radu', 'Sorina']})
df.merge(manageri, on='Departament', how='left')

## 8. Valori lipsă

**Cerere:** *„Câte valori lipsă are fiecare coloană? Elimină rândurile fără vârstă; în rest, completează numerele cu mediana și textul cu `necunoscut`.”*

Decizia contează: eliminarea pierde rânduri, completarea inventează valori. Cereți mereu și **numărul de rânduri înainte și după**.

In [ ]:
lipsa = pd.DataFrame({'Nume': ['Ana', 'Ion', 'Maria'], 'Varsta': [25, None, 22], 'Oras': ['Brasov', None, 'Cluj']})
print(lipsa.isna().sum())

lipsa.dropna(subset=['Varsta'])                                   # elimină rândurile fără vârstă
lipsa.fillna({'Varsta': lipsa['Varsta'].median(), 'Oras': 'necunoscut'})   # completează

## 9. Duplicate

**Cerere:** *„Câte rânduri duplicate sunt? Elimină-le și arată câte au rămas.”*

In [ ]:
dup = pd.concat([df, df.iloc[[0]]])      # ne facem un duplicat
print(dup.duplicated().sum(), 'duplicate')
dup.drop_duplicates().shape

## 10. Discretizare (împărțirea în grupe)

**Cerere 1, intervale fixe:** *„Împarte vârsta în grupele 0-25, 26-30, peste 30.”* → `pd.cut` cu margini.

**Cerere 2, grupe cu același număr de rânduri:** *„Împarte salariul în 3 grupe egale ca număr.”* → `pd.qcut`.

**Cerere 3, cu coloană de referință:** *„Împarte angajații după vârstă în 3 grupe, astfel încât fiecare grupă să însumeze aproximativ o treime din salarii.”* Se însumează salariile pe fiecare valoare distinctă a vârstei, se calculează ponderea cumulată, iar grupa se alege după ea. Toate rândurile cu aceeași vârstă rămân în aceeași grupă.

In [ ]:
df['Grupa_varsta'] = pd.cut(df['Varsta'], bins=[0, 25, 30, 100], labels=['<=25', '26-30', '>30'])
df['Grupa_salariu'] = pd.qcut(df['Salariu'], q=3, labels=['mic', 'mediu', 'mare'])
df[['Nume', 'Varsta', 'Grupa_varsta', 'Salariu', 'Grupa_salariu']]

In [ ]:
# discretizare cu coloană de referință
sal_pe_varsta = df.groupby('Varsta')['Salariu'].sum()             # salarii pe fiecare vârstă
proc_cum = sal_pe_varsta.cumsum() / sal_pe_varsta.sum()           # ponderea cumulată
df['Grupa'] = pd.cut(df['Varsta'].map(proc_cum), bins=[0, 1/3, 2/3, 1], labels=['G1', 'G2', 'G3'])

# verificare: ponderea fiecărei grupe din total (nu va fi exact 1/3: rândurile cu aceeași vârstă rămân împreună)
df.groupby('Grupa', observed=True).agg(varsta_min=('Varsta', 'min'), varsta_max=('Varsta', 'max'),
                                       proc_salarii=('Salariu', lambda s: s.sum() / df['Salariu'].sum()))

## Ce rămâne pentru alte module

Codificarea variabilelor categoriale (one-hot) și standardizarea (`StandardScaler`, `MinMaxScaler`) se folosesc la modelare, în modulele următoare. Regula atunci: **împărțiți train/test înainte** de orice transformare care învață din date, ca să evitați *data leakage*.

## Salvare

`df.to_csv("rezultat.csv", index=False)` sau `df.to_excel("rezultat.xlsx", index=False)`.

## Verificați-vă înțelegerea

1. Care este diferența dintre `merge` și `concat`? Dați un exemplu de cerere pentru fiecare.
2. De ce, după un `merge`, verificăm numărul de rânduri?
3. Formulați o cerere (în cuvintele voastre) pentru: „cele mai mari 5 produse după venit pe `lab-set4`”. Ce coloane și ce operații implică?
4. Când preferați `fillna` și când `dropna`?

| Exercițiu | Cererea mea | Ce am verificat |
|---|---|---|
| 3 |  |  |